In [8]:
"""
build_rag_index.py — сбор новой большой базы знаний для SmartHandyman.

Что делает:
- качает статьи и гайды с iFixit, WikiHow, Mastergrad и BobVila;
- режет тексты на чанки;
- считает эмбеддинги моделью paraphrase-multilingual-MiniLM-L12-v2;
- строит FAISS‑индекс и сохраняет в rag_store/:
    - index.faiss
    - chunks.pkl
    - metadata.pkl

Запуск из корня проекта (в venv):
    python build_rag_index.py
"""
from __future__ import annotations

# ─────────────────────────── Установка зависимостей (Kaggle / Colab) ───────────────────────────
import subprocess
import sys


def _pip(*packages: str) -> None:
    """Тихо ставит пакеты через pip текущего интерпретатора."""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        stdout=subprocess.DEVNULL,
    )


try:
    import faiss  # noqa: F401
except ImportError:
    print("[setup] Устанавливаю faiss-cpu…")
    _pip("faiss-cpu")

try:
    import sentence_transformers  # noqa: F401
except ImportError:
    print("[setup] Устанавливаю sentence-transformers…")
    _pip("sentence-transformers")

try:
    import bs4  # noqa: F401
except ImportError:
    print("[setup] Устанавливаю beautifulsoup4…")
    _pip("beautifulsoup4")

try:
    import tqdm  # noqa: F401
except ImportError:
    print("[setup] Устанавливаю tqdm…")
    _pip("tqdm")

try:
    import requests  # noqa: F401
except ImportError:
    print("[setup] Устанавливаю requests…")
    _pip("requests")

try:
    import numpy  # noqa: F401
except ImportError:
    print("[setup] Устанавливаю numpy…")
    _pip("numpy")

# ────────────────────────────────────────────────────────────────────────────────────────────────

import re
import time
import pickle
from collections import Counter
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

import faiss
from sentence_transformers import SentenceTransformer


PROJECT_ROOT = Path(__file__).parent if "__file__" in dir() else Path.cwd()
RAG_DIR = PROJECT_ROOT / "rag_store"
RAG_DIR.mkdir(exist_ok=True)

INDEX_PATH = RAG_DIR / "index.faiss"
CHUNKS_PATH = RAG_DIR / "chunks.pkl"
METADATA_PATH = RAG_DIR / "metadata.pkl"

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


# ─────────────────────────── Источники ───────────────────────────

# 1) iFixit
IFIXIT_BASE = "https://www.ifixit.com/api/2.0"
IFIXIT_HEADERS = {"User-Agent": "RAG-Handyman/cli-2.0"}


def ifixit_get(endpoint: str, params: dict | None = None) -> Optional[dict]:
    try:
        resp = requests.get(
            f"{IFIXIT_BASE}{endpoint}",
            headers=IFIXIT_HEADERS,
            params=params,
            timeout=20,
        )
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f"[iFixit] {endpoint}: {e}")
        return None


def get_ifixit_guides(category: str) -> List[dict]:
    data = ifixit_get(f"/wikis/CATEGORY/{category}")
    if not data:
        return []
    guides = data.get("guides", [])
    if not guides:
        for cl in data.get("category_lists", []):
            guides.extend(cl.get("guides", []))
    return guides


def get_ifixit_guide(guide_id: int) -> Optional[dict]:
    return ifixit_get(f"/guides/{guide_id}")


def parse_ifixit_guide(guide: dict) -> str:
    parts: List[str] = []
    title = guide.get("title", "")
    if title:
        parts.append(f"Guide: {title}")

    intro = guide.get("introduction_rendered") or guide.get("introduction", "")
    if intro:
        parts.append(BeautifulSoup(intro, "html.parser").get_text(" ").strip())

    for step in guide.get("steps", []):
        step_title = step.get("title", "")
        if step_title:
            parts.append(f"Step {step.get('orderby', '')}: {step_title}")
        for line in step.get("lines", []):
            text = line.get("text", "").strip()
            if not text:
                continue
            text = re.sub(r"\[([^\]]*?)\|?[^\]]*?\]", r"\1", text)
            text = re.sub(r"['\"][\"']{0,2}", "", text)
            parts.append(text)

    return "\n".join(filter(None, parts))


def fetch_ifixit_category(
    category: str, delay: float = 0.7, max_guides: int | None = None
) -> List[Dict]:
    guides_meta = get_ifixit_guides(category)
    if not guides_meta:
        print(f"[iFixit] пустая категория: {category}")
        return []

    results: List[Dict] = []
    iterable = guides_meta if max_guides is None else guides_meta[:max_guides]
    for meta in tqdm(iterable, desc=f"iFixit [{category}]"):
        guide_id = meta.get("guideid")
        if not guide_id:
            continue
        guide = get_ifixit_guide(guide_id)
        if not guide:
            continue
        text = parse_ifixit_guide(guide)
        if text:
            results.append(
                {
                    "text": text,
                    "source": f"https://www.ifixit.com/Guide/{guide_id}",
                    "title": guide.get("title", ""),
                    "provider": "ifixit",
                }
            )
        time.sleep(delay)
    print(f"[iFixit] {category}: {len(results)} гайдов")
    return results


# 2) WikiHow
WH_HEADERS = {"User-Agent": "RAG-Handyman/cli-2.0"}


def search_wikihow(query: str, max_results: int = 5) -> List[str]:
    try:
        resp = requests.get(
            "https://www.wikihow.com/wikiHowTo",
            headers=WH_HEADERS,
            params={"search": query, "ns": 0},
            timeout=20,
        )
        resp.raise_for_status()
    except Exception as e:
        print(f"[WikiHow] поиск '{query}': {e}")
        return []

    soup = BeautifulSoup(resp.text, "html.parser")
    urls: List[str] = []
    for a in soup.select("a.result_link"):
        href = a.get("href", "")
        if href.startswith("/"):
            href = "https://www.wikihow.com" + href
        if href and href not in urls:
            urls.append(href)
        if len(urls) >= max_results:
            break
    return urls


def parse_wikihow_article(url: str) -> Optional[Dict]:
    try:
        resp = requests.get(url, headers=WH_HEADERS, timeout=20)
        resp.raise_for_status()
    except Exception as e:
        print(f"[WikiHow] {url}: {e}")
        return None

    soup = BeautifulSoup(resp.text, "html.parser")
    parts: List[str] = []

    title_tag = soup.find("h1", class_="firstHeading") or soup.find("h1")
    title = title_tag.get_text(" ").strip() if title_tag else ""
    if title:
        parts.append(f"Article: {title}")

    intro = soup.find("div", id="intro")
    if intro:
        parts.append(intro.get_text(" ").strip())

    steps = soup.select("li.steps_list_2 .step") or soup.select(".step")
    for i, step in enumerate(steps, 1):
        for tag in step(["script", "style", "figure", "img", "video"]):
            tag.decompose()
        text = re.sub(r"\s+", " ", step.get_text(" ")).strip()
        if text:
            parts.append(f"Step {i}: {text}")

    for sec_id in ["tips", "warnings"]:
        sec = soup.find("div", id=sec_id)
        if sec:
            text = re.sub(r"\s+", " ", sec.get_text(" ")).strip()
            if text:
                parts.append(f"{sec_id.capitalize()}: {text}")

    if len(parts) < 2:
        print(f"[WikiHow] мало контента: {url}")
        return None

    return {
        "text": "\n".join(parts),
        "source": url,
        "title": title,
        "provider": "wikihow",
    }


def fetch_wikihow_topic(
    query: str, max_articles: int = 5, delay: float = 1.0
) -> List[Dict]:
    results: List[Dict] = []
    for url in tqdm(search_wikihow(query, max_articles), desc=f"WikiHow [{query}]"):
        article = parse_wikihow_article(url)
        if article:
            results.append(article)
        time.sleep(delay)
    print(f"[WikiHow] '{query}': {len(results)} статей")
    return results


# 3) Mastergrad
MG_BASE = "https://mastergrad.com"
MG_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X) AppleWebKit/537.36 Chrome/120 Safari/537.36",
    "Accept-Language": "ru-RU,ru;q=0.9",
}

MG_SECTIONS = [
    # бытовая техника
    "/forums/bytovaya-tehnika-i-elektronika/stiralnye-mashiny/",
    "/forums/bytovaya-tehnika-i-elektronika/holodilniki/",
    "/forums/bytovaya-tehnika-i-elektronika/posudomoechnye-mashiny/",
    "/forums/bytovaya-tehnika-i-elektronika/pylesosy/",
    "/forums/bytovaya-tehnika-i-elektronika/gazovye-plity-i-duhovki/",
    "/forums/bytovaya-tehnika-i-elektronika/mikrovolnovye-pechi/",
    "/forums/bytovaya-tehnika-i-elektronika/kondicionery/",
    # сантехника и отопление
    "/forums/otoplenie-vodosnabzhenie-kanalizaciya-i-santehnicheskoe-oborudovanie/santehnika-i-santehnicheskoe-oborudovanie/",
    "/forums/otoplenie-vodosnabzhenie-kanalizaciya-i-santehnicheskoe-oborudovanie/kanalizaciya/",
    "/forums/otoplenie-vodosnabzhenie-kanalizaciya-i-santehnicheskoe-oborudovanie/otoplenie/",
    "/forums/otoplenie-vodosnabzhenie-kanalizaciya-i-santehnicheskoe-oborudovanie/kotly-i-kotelnoe-oborudovanie/",
    "/forums/otoplenie-vodosnabzhenie-kanalizaciya-i-santehnicheskoe-oborudovanie/vodosnabzhenie/",
    "/forums/otoplenie-vodosnabzhenie-kanalizaciya-i-santehnicheskoe-oborudovanie/nasosy/",
    # электрика
    "/forums/elektrika-i-slabotochka/elektrika/",
    "/forums/elektrika-i-slabotochka/elektromontazh/",
    # ремонт и стройка
    "/forums/remont-i-otdelka/pol/",
    "/forums/remont-i-otdelka/steny/",
    "/forums/remont-i-otdelka/potolki/",
    "/forums/remont-i-otdelka/okna-i-dveri/",
]


def get_mg_thread_urls(section_path: str, max_threads: int = 15) -> List[str]:
    try:
        resp = requests.get(MG_BASE + section_path, headers=MG_HEADERS, timeout=20)
        resp.raise_for_status()
    except Exception as e:
        print(f"[Mastergrad] {section_path}: {e}")
        return []

    soup = BeautifulSoup(resp.text, "html.parser")
    urls: List[str] = []
    seen = set()

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if not href.startswith("http"):
            href = MG_BASE + href
        if re.search(r"/forums/t\d+", href) and href not in seen:
            seen.add(href)
            urls.append(href)
        if len(urls) >= max_threads:
            break

    return urls


def parse_mg_thread(url: str) -> Optional[Dict]:
    try:
        resp = requests.get(url, headers=MG_HEADERS, timeout=20)
        resp.raise_for_status()
    except Exception as e:
        print(f"[Mastergrad] {url}: {e}")
        return None

    soup = BeautifulSoup(resp.text, "html.parser")
    parts: List[str] = []

    h1 = soup.find("h1")
    title = h1.get_text(" ").strip() if h1 else ""
    if title:
        parts.append(f"Тема: {title}")

    posts_found = soup.select("div.post-body, div.pagetext")
    for post in posts_found:
        for tag in post(["script", "style", "blockquote", "aside"]):
            tag.decompose()
        text = re.sub(r"\s+", " ", post.get_text(" ")).strip()
        if len(text.split()) >= 20:
            parts.append(text)

    if len(parts) < 2:
        return None

    return {
        "text": "\n".join(parts),
        "source": url,
        "title": title,
        "provider": "mastergrad",
    }


def fetch_mastergrad_section(
    section_path: str, max_threads: int = 15, delay: float = 1.0
) -> List[Dict]:
    thread_urls = get_mg_thread_urls(section_path, max_threads=max_threads)
    if not thread_urls:
        print(f"[Mastergrad] нет тредов: {section_path}")
        return []

    results: List[Dict] = []
    label = section_path.rstrip("/").split("/")[-1][:30]
    for url in tqdm(thread_urls, desc=f"Mastergrad [{label}]"):
        doc = parse_mg_thread(url)
        if doc:
            results.append(doc)
        time.sleep(delay)

    print(f"[Mastergrad] {section_path}: {len(results)} тредов")
    return results


def fetch_mastergrad_all(max_threads_per_section: int = 10, delay: float = 1.0) -> List[Dict]:
    results: List[Dict] = []
    for section in MG_SECTIONS:
        results.extend(
            fetch_mastergrad_section(
                section, max_threads=max_threads_per_section, delay=delay
            )
        )
    print(f"[Mastergrad] всего тредов: {len(results)}")
    return results


# 4) Bob Vila
BV_BASE = "https://www.bobvila.com"
BV_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X) AppleWebKit/537.36 Chrome/120 Safari/537.36",
}

BV_CATEGORIES = [
    "/category/plumbing/",
    "/category/electrical/",
    "/category/repair-maintenance/",
    "/category/hvac/",
    "/category/doors/",
    "/category/windows/",
    "/category/flooring/",
    "/category/roofing/",
    "/category/bathroom/",
    "/category/kitchen/",
    "/category/heating-cooling/",
    "/category/appliances/",
    "/category/exterior/",
    "/category/basement/",
]


def get_bobvila_article_urls(category_path: str, max_results: int = 10) -> List[str]:
    try:
        resp = requests.get(BV_BASE + category_path, headers=BV_HEADERS, timeout=20)
        resp.raise_for_status()
    except Exception as e:
        print(f"[BobVila] {category_path}: {e}")
        return []

    soup = BeautifulSoup(resp.text, "html.parser")
    urls: List[str] = []
    seen = set()

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if not href.startswith("http"):
            href = BV_BASE + href
        if (
            "bobvila.com/articles/" in href
            and not href.endswith("/articles/")
            and href not in seen
        ):
            seen.add(href)
            urls.append(href)
        if len(urls) >= max_results:
            break

    return urls


def parse_bobvila_article(url: str) -> Optional[Dict]:
    try:
        resp = requests.get(url, headers=BV_HEADERS, timeout=20)
        resp.raise_for_status()
    except Exception as e:
        print(f"[BobVila] {url}: {e}")
        return None

    soup = BeautifulSoup(resp.text, "html.parser")
    h1 = soup.find("h1")
    if not h1:
        return None

    parts: List[str] = [f"Article: {h1.get_text(' ').strip()}"]
    content = (
        soup.select_one("div.article-body")
        or soup.select_one("div.comp.article-body")
        or soup.select_one("article")
        or soup.select_one("main")
    )
    if not content:
        return None

    for tag in content(["script", "style", "aside", "nav", "figure", "iframe"]):
        tag.decompose()

    for el in content.find_all(["h2", "h3", "p", "li"]):
        text = re.sub(r"\s+", " ", el.get_text(" ")).strip()
        if len(text) > 30:
            parts.append(text)

    if len(parts) < 2:
        return None

    return {
        "text": "\n".join(parts),
        "source": url,
        "title": parts[0].replace("Article: ", ""),
        "provider": "bobvila",
    }


def fetch_bobvila_categories(
    max_per_category: int = 10, delay: float = 1.0
) -> List[Dict]:
    results: List[Dict] = []
    for cat in BV_CATEGORIES:
        urls = get_bobvila_article_urls(cat, max_results=max_per_category)
        label = cat.strip("/").split("/")[-1][:25]
        for url in tqdm(urls, desc=f"BobVila [{label}]"):
            doc = parse_bobvila_article(url)
            if doc:
                results.append(doc)
            time.sleep(delay)
    print(f"[BobVila] всего статей: {len(results)}")
    return results


# ─────────────────────────── Чанкование ───────────────────────────

def chunk_text(
    text: str,
    chunk_size: int = 400,
    overlap: int = 80,
    min_chunk_words: int = 30,
) -> List[str]:
    """
    Разбивает текст на чанки с перекрытием.
    Короткий хвост (< min_chunk_words) присоединяется к предыдущему чанку.
    """

    words = text.split()
    if not words:
        return []

    step = chunk_size - overlap
    if step <= 0:
        raise ValueError("overlap должен быть меньше chunk_size")

    chunks: List[str] = []
    for i in range(0, len(words), step):
        chunk_words = words[i : i + chunk_size]
        if len(chunk_words) < min_chunk_words:
            if chunks:
                chunks[-1] += " " + " ".join(chunk_words)
            break
        chunks.append(" ".join(chunk_words))
    return chunks


def documents_to_chunks(
    documents: List[Dict], chunk_size: int = 400, overlap: int = 80
) -> tuple[list[str], list[dict]]:
    """Документы → (all_chunks, metadata)."""

    all_chunks: list[str] = []
    metadata: list[dict] = []

    for doc in documents:
        for chunk in chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap):
            all_chunks.append(chunk)
            metadata.append(
                {
                    "source": doc["source"],
                    "title": doc.get("title", ""),
                    "provider": doc.get("provider", "unknown"),
                }
            )

    print(f"Итого чанков: {len(all_chunks)}")

    providers = Counter(m["provider"] for m in metadata)
    print("Чанков по источникам:")
    for p, n in providers.items():
        print(f"  {p:12s}: {n}")

    return all_chunks, metadata


# ─────────────────────────── Main ───────────────────────────

def main() -> None:
    # Параметры масштаба (можно уменьшить, если хочешь быстрее)
    max_ifixit_guides_per_category = None  # все гайды, без ограничений
    max_wikihow_articles_per_query = 10
    max_mg_threads_per_section = 30
    max_bobvila_per_category = 25

    print("PROJECT_ROOT:", PROJECT_ROOT)
    print("RAG_DIR     :", RAG_DIR)
    print("Модель      :", EMBEDDING_MODEL)

    all_documents: List[Dict] = []

    ifixit_categories = [
        "Washing_Machine",
        "Dryer",
        "Dishwasher",
        "Refrigerator",
        "Freezer",
        "Oven",
        "Microwave_Oven",
        "Electric_Stove",
        "Gas_Stove",
        "Range_Hood",
        "Air_Conditioner",
        "Water_Heater",
        "Toilet",
        "Faucet",
        "Shower",
        "Sink",
        "Heat_Pump",
        "Furnace",
        "Lamp",
        "Ceiling_Fan",
        "Pipe",
        "Plumbing",
        "Bathroom",
        "Kitchen",
        "Boiler",
        "Thermostat",
        "Smoke_Detector",
        "Door_Lock",
        "Garage_Door",
        "Vacuum_Cleaner",
        "Coffee_Maker",
    ]

    wikihow_queries = [
        "washing machine not draining water",
        "washing machine not spinning clothes",
        "washing machine leaking from bottom",
        "dryer not heating up",
        "refrigerator not cooling but freezer works",
        "dishwasher not draining",
        "fix leaking faucet dripping",
        "fix running toilet constantly",
        "unclog toilet without plunger",
        "unclog bathroom sink drain",
        "unclog shower drain hair",
        "water heater not producing hot water",
        "circuit breaker keeps tripping",
        "electrical outlet not working",
        "ceiling light flickering",
        "air conditioner not blowing cold air",
        "furnace not heating house",
        "how to fix cracked pipe bathroom",
        "how to repair leaking pipe wall",
        "pipe burst repair temporary fix",
        "how to patch copper pipe leak",
        "how to fix pinhole leak in pipe",
        "how to repair pvc pipe crack",
        "how to stop pipe leak without replacing",
        "pipe joint leaking repair",
        "water leak behind wall how to fix",
        "radiator not heating up",
        "boiler not working no hot water",
        "thermostat not working fix",
        "smoke detector beeping how to fix",
        "garage door not opening fix",
        "door lock stuck how to fix",
        "ceiling fan wobbling fix",
        "light switch not working fix",
        "water pressure low fix",
        "toilet clogged badly fix",
    ]

    print("\n=== iFixit ===")
    for category in ifixit_categories:
        all_documents.extend(
            fetch_ifixit_category(
                category,
                delay=0.7,
                max_guides=max_ifixit_guides_per_category,
            )
        )

    print("\n=== WikiHow ===")
    for query in wikihow_queries:
        all_documents.extend(
            fetch_wikihow_topic(
                query,
                max_articles=max_wikihow_articles_per_query,
                delay=1.0,
            )
        )

    print("\n=== Mastergrad ===")
    all_documents.extend(
        fetch_mastergrad_all(
            max_threads_per_section=max_mg_threads_per_section,
            delay=1.0,
        )
    )

    print("\n=== BobVila ===")
    all_documents.extend(
        fetch_bobvila_categories(
            max_per_category=max_bobvila_per_category,
            delay=1.0,
        )
    )

    print("\nВсего документов:", len(all_documents))
    if not all_documents:
        print("Документов нет — ничего не сохраняю.")
        return

    # Чанки
    all_chunks, metadata = documents_to_chunks(
        all_documents, chunk_size=400, overlap=80
    )

    # Эмбеддинги
    print("\nСчитаю эмбеддинги (это может занять несколько минут)...")
    model = SentenceTransformer(EMBEDDING_MODEL)
    embeddings = model.encode(
        all_chunks,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    print("Эмбеддинги:", embeddings.shape)

    # FAISS‑индекс
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    print("Проиндексировано векторов:", index.ntotal)

    print("\nДокументов по источникам:")
    providers_stat = Counter(d["provider"] for d in all_documents)
    for p, n in providers_stat.items():
        print(f"  {p:12s}: {n}")

    faiss.write_index(index, str(INDEX_PATH))
    with CHUNKS_PATH.open("wb") as f:
        pickle.dump(all_chunks, f)
    with METADATA_PATH.open("wb") as f:
        pickle.dump(metadata, f)

    print("\nГотово!")
    print("  index  ->", INDEX_PATH)
    print("  chunks ->", CHUNKS_PATH)
    print("  meta   ->", METADATA_PATH)
    print("\nТеперь можно запустить:")
    print("  python test_rag.py")
    print("и посмотреть, какие источники находит RAG.")


if __name__ == "__main__":
    main()

PROJECT_ROOT: /kaggle/working
RAG_DIR     : /kaggle/working/rag_store
Модель      : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

=== iFixit ===


iFixit [Washing_Machine]:   0%|          | 0/29 [00:00<?, ?it/s]

[iFixit] Washing_Machine: 29 гайдов


iFixit [Dryer]:   0%|          | 0/16 [00:00<?, ?it/s]

[iFixit] Dryer: 16 гайдов


...

[BobVila] всего статей: 107

Всего документов: 757
Итого чанков: 3399
Чанков по источникам:
  ifixit      : 186
  wikihow     : 2023
  mastergrad  : 207
  bobvila     : 983

Считаю эмбеддинги (это может занять несколько минут)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/54 [00:00<?, ?it/s]

Эмбеддинги: (3399, 384)
Проиндексировано векторов: 3399

Документов по источникам:
  ifixit      : 203
  wikihow     : 338
  mastergrad  : 109
  bobvila     : 107

Готово!
  index  -> /kaggle/working/rag_store/index.faiss
  chunks -> /kaggle/working/rag_store/chunks.pkl
  meta   -> /kaggle/working/rag_store/metadata.pkl

Теперь можно запустить:
  python test_rag.py
и посмотреть, какие источники находит RAG.
